In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import gc # Garbage Collector para liberar RAM

In [ ]:
import os
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd

/content/drive/MyDrive/nids-mitre


In [ ]:
# --- CONFIGURATION ---
CSV_PATH = "/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv"
BASE_OUTPUT_DIR = "./dataset_processed"
TIME_WINDOW_SEC = 30  # Each graph represents 30 seconds
CHUNK_SIZE = 1000000   # Rows to read at a time in RAM

In [ ]:
cicids2018v3_wed2802_orig = pd.read_csv(CSV_PATH)
cicids2018v3_wed2802 = cicids2018v3_wed2802_orig.copy()
# Convert 'FLOW_START_TIME' to datetime type after loading
cicids2018v3_wed2802['FLOW_START_TIME'] = pd.to_datetime(cicids2018v3_wed2802['FLOW_START_TIME'])

timestamps = cicids2018v3_wed2802['FLOW_START_TIME']
GLOBAL_START_TIME = timestamps.min()
GLOBAL_END_TIME = timestamps.max()
TOTAL_DURATION = GLOBAL_END_TIME - GLOBAL_START_TIME

# Limits for Train/Val/Test
LIMIT_TRAIN = GLOBAL_START_TIME + (TOTAL_DURATION * 0.70)
LIMIT_VAL = GLOBAL_START_TIME + (TOTAL_DURATION * 0.85)

print(f"Global start time: {GLOBAL_START_TIME}")
print(f"Global end time: {GLOBAL_END_TIME}")

print(f"Limit train: {LIMIT_TRAIN}")
print(f"Limit val: {LIMIT_VAL}")

Global start time: 2018-02-28 00:12:55.854000
Global end time: 2018-02-28 23:59:59.321000
Limit train: 2018-02-28 16:51:52.280900
Limit val: 2018-02-28 20:25:55.800950


In [ ]:
NUMERIC_FEATURES = [
    'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
    'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT',
    'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_AVG',
    'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_STDDEV',
    'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
    'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_PKTS',
    'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
    'TCP_FLAGS', 'MIN_TTL', 'MAX_TTL'
]

CATEGORICAL_FEATURES = [
    'PROTOCOL', 'L4_DST_PORT'
]

In [ ]:
def apply_port_roles(df):
    """Crea columnas One-Hot manuales para los roles de puerto"""
    # 1. Web
    df['Role_Web'] = df['L4_DST_PORT'].isin([80, 443, 8080, 8443, 81, 3128, 8545]).astype(int)
    # 2. Admin
    df['Role_Admin'] = df['L4_DST_PORT'].isin([22, 222, 2222, 23, 2323, 3389, 3390, 3394, 5900, 5901, 5555, 21, 2131]).astype(int)
    # 3. Windows
    df['Role_Win'] = df['L4_DST_PORT'].isin([445, 135, 137, 138, 139]).astype(int)
    # 4. DNS/Infra
    df['Role_Infra'] = df['L4_DST_PORT'].isin([53, 5355, 67, 547, 123, 1900, 5060]).astype(int)
    # 5. DB
    df['Role_DB'] = df['L4_DST_PORT'].isin([1433, 3306, 5432, 6379, 27017]).astype(int)
    # 6. Privileged Low (Resto < 1024)
    # Excluimos los que ya están arriba para no duplicar lógica complicada,
    # simplificamos: si es < 1024 y no es ninguno de los anteriores.
    # Para RF es suficiente con marcar si es < 1024.
    df['Role_LowPriv'] = ((df['L4_DST_PORT'] < 1024) & (df['Role_Web']==0) & (df['Role_Admin']==0) & (df['Role_Win']==0) & (df['Role_Infra']==0)).astype(int)
    df['Role_HighPriv'] = (df['L4_DST_PORT'] >= 1024).astype(int)

    return df

In [ ]:
def process_chunk(chunk):
    # 1. Limpieza 0.0.0.0
    chunk = chunk[ (chunk['IPV4_SRC_ADDR'] != '0.0.0.0') & (chunk['IPV4_DST_ADDR'] != '0.0.0.0') ].copy()

    # 2. Fecha
    chunk['FLOW_START_TIME'] = pd.to_datetime(chunk['FLOW_START_TIME'])

    # 3. Log1p en numéricas
    chunk[NUMERIC_FEATURES] = np.log1p(chunk[NUMERIC_FEATURES].astype(float))

    # 4. Ingeniería de Puerto (Crear columnas explícitas)
    chunk = apply_port_roles(chunk)

    # 5. Ingeniería de Protocolo (One Hot Simple)
    chunk['Proto_TCP'] = (chunk['PROTOCOL'] == 6).astype(int)
    chunk['Proto_UDP'] = (chunk['PROTOCOL'] == 17).astype(int)
    chunk['Proto_ICMP_ICMPv6'] = ((chunk['PROTOCOL'] == 1) | (chunk['PROTOCOL'] == 58)).astype(int)
    chunk['Proto_IGMP'] = (chunk['PROTOCOL'] == 2).astype(int)
    chunk['Proto_Other'] = ((chunk['PROTOCOL'] != 6) & (chunk['PROTOCOL'] != 17) & (chunk['PROTOCOL'] != 1) & (chunk['PROTOCOL'] != 58) & (chunk['PROTOCOL'] != 2)).astype(int)

    # 6. Target
    chunk['Target'] = chunk['Attack'].apply(lambda x: 1 if x == 'Infilteration' else 0)

    return chunk

In [ ]:
# --- CARGA Y SPLIT ---
print("Leyendo CSV y separando por tiempo...")
train_list = []
val_list = []
test_list = []

# Iterar CSV (Esto ahorra RAM, procesamos y guardamos lo justo)
for i, chunk in enumerate(pd.read_csv(CSV_PATH, usecols=NUMERIC_FEATURES + ['L4_DST_PORT', 'PROTOCOL', 'Attack', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'FLOW_START_TIME'], chunksize=CHUNK_SIZE)):

    df_proc = process_chunk(chunk)

    # Separar por tiempo
    mask_train = df_proc['FLOW_START_TIME'] < LIMIT_TRAIN
    mask_val = (df_proc['FLOW_START_TIME'] >= LIMIT_TRAIN) & (df_proc['FLOW_START_TIME'] < LIMIT_VAL)
    mask_test = df_proc['FLOW_START_TIME'] >= LIMIT_VAL

    if mask_train.sum() > 0:
        train_list.append(df_proc[mask_train])

    if mask_val.sum() > 0:
        val_list.append(df_proc[mask_val])

    if mask_test.sum() > 0:
        test_list.append(df_proc[mask_test])

    print(f"Chunk {i} procesado. Train rows: {mask_train.sum()}, Val rows: {mask_val.sum()}, Test rows: {mask_test.sum()}")

# Concatenar
print("Concatenando DataFrames...")
df_train = pd.concat(train_list, ignore_index=True)
df_val = pd.concat(val_list, ignore_index=True)
df_test = pd.concat(test_list, ignore_index=True)

# Limpiar listas para liberar RAM
del train_list, val_list, test_list
gc.collect()

# Definir Columnas Finales para el Modelo
# (Numéricas + Las nuevas binarias que creamos)
FEATS = NUMERIC_FEATURES + ['Role_Web', 'Role_Admin', 'Role_Win', 'Role_Infra', 'Role_DB', 'Role_LowPriv', 'Role_HighPriv', 'Proto_TCP', 'Proto_UDP', 'Proto_ICMP_ICMPv6', 'Proto_IGMP', 'Proto_Other']

X_train = df_train[FEATS]
y_train = df_train['Target']
X_val = df_val[FEATS]
y_val = df_val['Target']

print(f"Dimensiones finales: Train {X_train.shape}, Val {X_val.shape}")
print(f"Ataques en Train: {y_train.sum()}")
print(f"Ataques en Val: {y_val.sum()}")

Leyendo CSV y separando por tiempo...
Chunk 0 procesado. Train rows: 994926, Val rows: 0, Test rows: 0
Chunk 1 procesado. Train rows: 46719, Val rows: 788681, Test rows: 159306
Chunk 2 procesado. Train rows: 0, Val rows: 0, Test rows: 32375
Concatenando DataFrames...
Dimensiones finales: Train (1041645, 32), Val (788681, 32)
Ataques en Train: 49564
Ataques en Val: 31329


In [ ]:
# Guardar datos procesados del baseline
df_train.to_parquet("./dataset_processed/train_tabular.parquet")
df_val.to_parquet("./dataset_processed/val_tabular.parquet")
df_test.to_parquet("./dataset_processed/test_tabular.parquet")
# (Usa parquet que es más rápido y comprime mejor que CSV, o pickle si prefieres)

In [ ]:
# --- ENTRENAMIENTO ---
print("\nEntrenando Random Forest...")
# class_weight='balanced' es CLAVE aquí para simular lo que buscábamos con pos_weight
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,       # Limitamos profundidad para que no memorice ruido
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

# --- EVALUACIÓN ---
print("\nEvaluando en Validación...")
preds = rf.predict(X_val)
probs = rf.predict_proba(X_val)[:, 1]

print("--- Reporte de Clasificación ---")
print(classification_report(y_val, preds))

print("-- Matriz de Confusión ---")
print(confusion_matrix(y_val, preds))

print(f"F1 Score (Clase 1): {f1_score(y_val, preds):.4f}")

# Feature Importance (Para ver qué está mirando)
importances = pd.Series(rf.feature_importances_, index=FEATS).sort_values(ascending=False)
print("\n--- Top 10 Features más importantes ---")
print(importances.head(10))


Entrenando Random Forest...

Evaluando en Validación...
--- Reporte de Clasificación ---
              precision    recall  f1-score   support

           0       0.99      0.68      0.81    757352
           1       0.10      0.86      0.18     31329

    accuracy                           0.69    788681
   macro avg       0.55      0.77      0.49    788681
weighted avg       0.96      0.69      0.78    788681

-- Matriz de Confusión ---
[[515438 241914]
 [  4414  26915]]
F1 Score (Clase 1): 0.1793

--- Top 10 Features más importantes ---
OUT_BYTES          0.098944
IN_BYTES           0.097024
TCP_FLAGS          0.093019
MIN_IP_PKT_LEN     0.084104
MAX_IP_PKT_LEN     0.075419
TCP_WIN_MAX_IN     0.057463
OUT_PKTS           0.053690
TCP_WIN_MAX_OUT    0.051204
IN_PKTS            0.040140
MIN_TTL            0.037286
dtype: float64


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Obtener probabilidades (0.0 a 1.0) en lugar de clases duras (0 o 1)
y_probs = rf.predict_proba(X_val)[:, 1]

print("--- Buscando el Umbral Óptimo ---")
best_th = 0.5
best_f1 = 0.0

# Probamos cortes más estrictos
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

for th in thresholds:
    preds_th = (y_probs >= th).astype(int)

    f1 = f1_score(y_val, preds_th)
    prec = precision_score(y_val, preds_th)
    rec = recall_score(y_val, preds_th)

    print(f"Umbral {th:.2f} -> F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_th = th

print(f"\n✅ Mejor configuración Baseline: Umbral {best_th} con F1 {best_f1:.4f}")

--- Buscando el Umbral Óptimo ---
Umbral 0.50 -> F1: 0.1793 | Precision: 0.1001 | Recall: 0.8591
Umbral 0.60 -> F1: 0.1979 | Precision: 0.1146 | Recall: 0.7250
Umbral 0.70 -> F1: 0.3130 | Precision: 0.3236 | Recall: 0.3032
Umbral 0.80 -> F1: 0.3368 | Precision: 0.4854 | Recall: 0.2578
Umbral 0.90 -> F1: 0.3282 | Precision: 0.5842 | Recall: 0.2282
Umbral 0.95 -> F1: 0.3049 | Precision: 0.5900 | Recall: 0.2056

✅ Mejor configuración Baseline: Umbral 0.8 con F1 0.3368


In [ ]:
import pandas as pd
import os
import joblib
from datetime import datetime, timezone, timedelta
from sklearn.metrics import fbeta_score, average_precision_score, precision_recall_curve, auc

class ExperimentManager:
    def __init__(self, log_file="./logs/experiments_log.csv", model_dir="./saved_models"):
        self.log_file = log_file
        self.model_dir = model_dir
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        os.makedirs(model_dir, exist_ok=True)

    def log_experiment(self, model_name, params, metrics, model_object=None):
        """
        Registra un experimento en el CSV y guarda el modelo opcionalmente.

        params: dict con hiperparámetros (ej: {'lr': 0.001, 'pos_weight': 5})
        metrics: dict con resultados (ej: {'f1': 0.3, 'auc_pr': 0.4})
        """
        # 1. Crear registro
        tz = timezone(timedelta(hours=-3))  # Argentina time zone
        now = datetime.now(tz)
        entry = {
            'timestamp': now.strftime("%Y-%m-%d %H:%M:%S"),
            'model_name': model_name,
            **params,  # Desempaqueta hiperparámetros como columnas
            **metrics  # Desempaqueta métricas como columnas
        }

        # 2. Guardar en CSV
        df_new = pd.DataFrame([entry])
        if os.path.exists(self.log_file):
            df_new.to_csv(self.log_file, mode='a', header=False, index=False)
        else:
            df_new.to_csv(self.log_file, mode='w', header=True, index=False)

        print(f"📝 Experimento registrado en {self.log_file}")

        # 3. Guardar Modelo (si se provee)
        if model_object:
            # Nombre de archivo único basado en timestamp y métrica clave
            metric_key = 'AUC-PR' if 'AUC-PR' in metrics else 'F1'
            metric_val = metrics.get(metric_key, 0)
            filename = f"{model_name}_{now.strftime('%Y%m%d_%H%M')}_{metric_key}_{metric_val:.4f}"

            if 'sklearn' in str(type(model_object)):
                joblib.dump(model_object, os.path.join(self.model_dir, f"{filename}.joblib"))
            else:
                # Asumimos PyTorch
                import torch
                torch.save(model_object.state_dict(), os.path.join(self.model_dir, f"{filename}.pth"))

            print(f"💾 Modelo guardado como: {filename}")

# Instancia global
exp_manager = ExperimentManager()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, confusion_matrix
import numpy as np

def calculate_metrics(y_true, y_probs, probs_threshold=0.5):
    preds = (y_probs > probs_threshold).astype(int)

    # Calculate standard metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)

    # Calculate other metrics
    f2 = fbeta_score(y_val, preds, beta=2)
    ap = average_precision_score(y_val, y_probs)
    roc = roc_auc_score(y_val, y_probs)

    # Calculate False Positive Rate (FPR)
    # TN = True Negative, FP = False Positive
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2, "AUC-PR": ap, "AUC-ROC": roc,
        "FPR": fpr, "TP": tp, "FP": fp, "TN": tn, "FN": fn, "Total_Flows": len(y_true)
    }

In [ ]:
y_probs_rf = rf.predict_proba(X_val)[:, 1]

metrics_rf = calculate_metrics(y_val, y_probs_rf)

In [ ]:
metrics_rf

{'Precision': 0.10011940676043135,
 'Recall': 0.8591081745347761,
 'F1': 0.1793388815223982,
 'F2': 0.34143525859772417,
 'AUC-PR': np.float64(0.28610657832937725),
 'AUC-ROC': np.float64(0.8536069263173388),
 'FPR': np.float64(0.3194208241346164),
 'TP': np.int64(26915),
 'FP': np.int64(241914),
 'TN': np.int64(515438),
 'FN': np.int64(4414),
 'Total_Flows': 788681}

In [ ]:
metrics_rf = {
    k: float(v) if isinstance(v, (np.floating, np.integer)) else v
    for k, v in metrics_rf.items()
}
print(metrics_rf)

{'Precision': 0.10011940676043135, 'Recall': 0.8591081745347761, 'F1': 0.1793388815223982, 'F2': 0.34143525859772417, 'AUC-PR': 0.286106578316185, 'AUC-ROC': 0.853606926275193, 'FPR': 0.3194208241346164, 'TP': 26915.0, 'FP': 241914.0, 'TN': 515438.0, 'FN': 4414.0, 'Total_Flows': 788681}


In [ ]:
# # Asumiendo que 'rf', 'X_val', 'y_val' existen de tu celda anterior

# # 1. Obtener probabilidades
# y_probs_rf = rf.predict_proba(X_val)[:, 1]

# # 2. Calcular F2 y AUC-PR
# f2 = fbeta_score(y_val, (y_probs_rf > 0.5).astype(int), beta=2)
# precision, recall, _ = precision_recall_curve(y_val, y_probs_rf)
# auc_pr = auc(recall, precision) # O usar average_precision_score

# metrics_rf = {
#     'f1': f1_score(y_val, (y_probs_rf > 0.5).astype(int)),
#     'f2': f2,
#     'auc_pr': auc_pr,
#     'recall': recall_score(y_val, (y_probs_rf > 0.5).astype(int)),
#     'precision': precision_score(y_val, (y_probs_rf > 0.5).astype(int))
# }

params_rf = {
    'type': 'Baseline_RF',
    'n_estimators': 100,
    'class_weight': 'balanced',
    'max_depth': 15,
    'threshold': 0.5
}

# 3. Registrar y Guardar
exp_manager.log_experiment("RandomForest_Baseline", params_rf, metrics_rf, rf)

📝 Experimento registrado en ./logs/experiments_log.csv


NameError: name 'now' is not defined